# Stage 2 — Daily Macro Baseline

Ridge regression: daily 30y close ~ Bloomberg macro factors.  
Produces `macro_fair_30y` (fitted) and `macro_resid_daily` (residual).  
The residual is a macro-adjusted feature used in Stage 6.

> **Leakage note:** `macro_baseline()` is refit on train-only rows inside every CV fold in Stage 6.
> The full-sample fit here is for exploratory inspection only.

**Reads:** `data/cache/daily_curve.parquet`  
**Writes:** `data/cache/daily_stage2.parquet`

Refs: Ang-Piazzesi (2003); Diebold-Rudebusch-Aruoba (2006); Adrian-Crump-Moench (2013).

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from config import CACHE_DIR, MACRO_COLS
from src.macro import macro_baseline

daily_df = pd.read_parquet(CACHE_DIR / 'daily_curve.parquet')
print(f'Daily rows: {len(daily_df)}  |  macro cols available: '
      f'{[c for c in MACRO_COLS if c in daily_df.columns]}')

## 2a. Full-sample fit (exploratory only)

Using all rows as a train_mask of all-True gives a full-sample Ridge fit.

In [ ]:
train_mask_all = np.ones(len(daily_df), dtype=bool)
daily_sub = macro_baseline(daily_df, MACRO_COLS, train_mask_all)

print('Columns added:', ['macro_fair_30y', 'macro_resid_daily'])
daily_sub[['macro_fair_30y', 'macro_resid_daily']].describe()

## 2b. Diagnostics

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(13, 6), sharex=True)

if 'y30_close' in daily_sub.columns:
    daily_sub['y30_close'].plot(ax=axes[0], label='y30_close (actual)', color='steelblue')
daily_sub['macro_fair_30y'].plot(ax=axes[0], label='macro_fair_30y (Ridge fit)',
                                 color='tomato', linestyle='--')
axes[0].set_title('30Y yield vs macro fair value')
axes[0].legend()

daily_sub['macro_resid_daily'].plot(ax=axes[1], color='gray')
axes[1].axhline(0, color='black', linewidth=0.8)
axes[1].set_title('macro_resid_daily (term-premium / supply proxy)')

plt.tight_layout()
plt.show()

In [ ]:
# Residual autocorrelation
from pandas.plotting import autocorrelation_plot
autocorrelation_plot(daily_sub['macro_resid_daily'].dropna())
plt.title('ACF of macro_resid_daily')
plt.show()

## 2c. Persist

In [ ]:
daily_sub.to_parquet(CACHE_DIR / 'daily_stage2.parquet')
print('Saved → cache/daily_stage2.parquet')